<a href="https://colab.research.google.com/github/Skulfre/Machine-Learning/blob/main/CIFAR100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
root_path = 'data/'
train_dataset = torchvision.datasets.CIFAR10(root=root_path,
                                             train=True,
                                             transform=transforms.ToTensor(),
                                             download=True)
test_dataset = torchvision.datasets.CIFAR10(root=root_path,
                                            train=False,
                                            transform=transforms.ToTensor(),
                                            download=True)
len(train_dataset), len(test_dataset)

In [ ]:
import os
BATCH_SIZE = 32

train_dataloader = DataLoader(dataset=train_dataset,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                              num_workers=os.cpu_count())

test_dataloader = DataLoader(dataset=test_dataset,
                             batch_size=BATCH_SIZE,
                             shuffle=False,
                             num_workers=os.cpu_count())

len(train_dataloader), len(test_dataloader)

In [ ]:
class_names = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

class_names

In [ ]:
image, label = next(iter(train_dataloader))
image.shape

In [ ]:
class TinyVGG(nn.Module):
  def __init__(self, in_features, hidden_layers, out_features):
    super().__init__()

    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=in_features, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_layers*8*8, out_features=out_features)
    )

  def forward(self, x):
    return self.classifier(self.conv_block_2(self.conv_block_1(x)))

model_0 = TinyVGG(in_features=3, hidden_layers=32, out_features=len(class_names))

In [ ]:
rand_tensor = torch.randn(32, 3, 32, 32)
output = model_0(rand_tensor)
output.shape

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(), lr=0.0005)

In [ ]:
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = (correct / len(y_pred)) * 100
    return acc

In [ ]:
def train_step(model, dataloader, loss_fn, accuracy_fn, optimizer, device):

  train_loss, train_acc = 0, 0
  model.to(device)
  model.train()

  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)
    pred_logit = model(X)
    pred_prob = torch.softmax(pred_logit, dim=1)
    pred_label = torch.argmax(pred_prob, dim=1)

    loss = loss_fn(pred_logit, y)
    train_loss += loss.item()

    accuracy = accuracy_fn(y, pred_label)
    train_acc += accuracy

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  train_loss /= len(dataloader)
  train_acc /= len(dataloader)

  return train_loss, train_acc

In [ ]:
def test_step(model, dataloader, loss_fn, accuracy_fn, device):

  test_loss, test_acc = 0, 0
  model.to(device)

  model.eval()
  with torch.inference_mode():
    for (X, y) in dataloader:
      X, y = X.to(device), y.to(device)
      pred_logit = model(X)
      pred_prob = torch.softmax(pred_logit, dim=1)
      pred_label = torch.argmax(pred_prob, dim=1)

      loss = loss_fn(pred_logit, y)
      test_loss += loss.item()

      accuracy = accuracy_fn(y, pred_label)
      test_acc += accuracy

  test_loss /= len(dataloader)
  test_acc /= len(dataloader)

  return test_loss, test_acc

In [ ]:
from tqdm.auto import tqdm

epochs = 10

for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train_step(model_0, train_dataloader, loss_fn, accuracy_fn, optimizer, device)
    test_loss, test_acc = test_step(model_0, test_dataloader, loss_fn, accuracy_fn, device)

    print(f"Epoch: {epoch} | Train loss: {train_loss:.4f}, Train acc: {train_acc:.2f}% | Test loss: {test_loss:.4f}, Test acc: {test_acc:.2f}%")